# Setup

In [ ]:
!pip install datasets transformers tqdm

In [ ]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# Config

In [ ]:
class Config:
    vocab_size = 50257
    block_size = 1024
    n_layer = 4
    n_head = 4
    n_embd = 256
    dropout = 0.1

# Load Dataset + Tokenizer

In [ ]:
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Tokenize + Chunk

In [ ]:
def tokenize(example):
    return tokenizer(example["text"])

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])

def group_texts(examples):
    block_size = Config.block_size
    concatenated = sum(examples["input_ids"], [])
    total_length = (len(concatenated) // block_size) * block_size

    input_ids = [
        concatenated[i:i + block_size]
        for i in range(0, total_length, block_size)
    ]

    return {"input_ids": input_ids, "labels": input_ids.copy()}

lm_datasets = tokenized.map(
    group_texts,
    batched=True,
    remove_columns=tokenized["train"].column_names,
)

# DataLoader

In [ ]:
def collate(batch):
    input_ids = torch.tensor([x["input_ids"] for x in batch])
    labels = torch.tensor([x["labels"] for x in batch])
    return input_ids, labels

train_loader = torch.utils.data.DataLoader(
    lm_datasets["train"],
    batch_size=8,
    shuffle=True,
    collate_fn=collate
)

val_loader = torch.utils.data.DataLoader(
    lm_datasets["validation"],
    batch_size=8,
    shuffle=False,
    collate_fn=collate
)

# Model Components

### Attention (Modular)

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.proj = nn.Linear(config.n_embd, config.n_embd)

        self.dropout = nn.Dropout(config.dropout)

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(config.block_size, config.block_size))
            .unsqueeze(0).unsqueeze(0)
        )

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))

        att = torch.softmax(att, dim=-1)
        att = self.dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        return self.proj(y)

### Feedforward

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)

### Transformer Block

In [ ]:
class Block(nn.Module):
    def __init__(self, config, attention_cls):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = attention_cls(config)

        self.ln2 = nn.LayerNorm(config.n_embd)
        self.ff = FeedForward(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

### Full Model

In [ ]:
class TransformerModel(nn.Module):
    def __init__(self, config, attention_cls):
        super().__init__()

        self.token_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Parameter(torch.zeros(1, config.block_size, config.n_embd))

        self.blocks = nn.ModuleList([
            Block(config, attention_cls)
            for _ in range(config.n_layer)
        ])

        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok = self.token_emb(idx)
        pos = self.pos_emb[:, :T, :]

        x = tok + pos

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits[:, :-1, :].contiguous().view(-1, logits.size(-1)),
                targets[:, 1:].contiguous().view(-1)
            )

        return logits, loss

# Initialise Model

In [ ]:
model = TransformerModel(Config, CausalSelfAttention).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# Training Loop

In [ ]:
def evaluate(model):
    model.eval()
    losses = []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            _, loss = model(x, y)
            losses.append(loss.item())

    model.train()
    return sum(losses) / len(losses)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import random
import numpy as np
import pandas as pd

seed = 42
epochs = 15
metrics_path = "/content/drive/MyDrive/baseline_seeded_15epoch_metrics.csv"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(seed)

model = TransformerModel(Config, CausalSelfAttention).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

training_metrics = []

for epoch in range(epochs):
    model.train()
    pbar = tqdm(train_loader, desc=f"seed {seed} epoch {epoch}")

    total_loss = 0.0
    total_tokens = 0

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    for x, y in pbar:
        x, y = x.to(device), y.to(device)

        _, loss = model(x, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_tokens = y[:, 1:].numel()
        total_tokens += batch_tokens
        total_loss += loss.item()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    epoch_time = time.perf_counter() - start_time

    train_loss = total_loss / len(train_loader)
    throughput = total_tokens / epoch_time

    val_loss = evaluate(model)
    perplexity = math.exp(val_loss)

    peak_gpu_memory_mb = (
        torch.cuda.max_memory_allocated() / (1024 ** 2)
        if torch.cuda.is_available()
        else 0.0
    )

    epoch_metrics = {
        "seed": seed,
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_perplexity": perplexity,
        "epoch_time_sec": epoch_time,
        "throughput_tokens_per_sec": throughput,
        "peak_gpu_memory_mb": peak_gpu_memory_mb,
    }

    training_metrics.append(epoch_metrics)

    pd.DataFrame(training_metrics).to_csv(metrics_path, index=False)

    print(f"Epoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Perplexity: {perplexity:.2f}")
    print(f"Epoch Time: {epoch_time:.2f}s")
    print(f"Throughput: {throughput:.2f} tokens/sec")
    print(f"Peak GPU Memory: {peak_gpu_memory_mb:.2f} MB")

training_df = pd.DataFrame(training_metrics)
display(training_df)

# Sanity Check (Generation)

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=50):
    model.eval()
    tokens = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    for _ in range(max_new_tokens):
        tokens = tokens[:, -Config.block_size:]
        logits, _ = model(tokens)

        next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
        tokens = torch.cat([tokens, next_token], dim=1)

    return tokenizer.decode(tokens[0])

In [ ]:
print(generate(model, tokenizer, "The meaning of life is"))